# GC-LSTM-GhostNet â€” Step 2 smoke test

This notebook checks the preserved CIC-DDoS2019 Parquet schema, group-first 70/80 splits, leakage assertions, and train-only preprocessing. It does not train the final model.

In [ ]:
from pathlib import Path
import base64
import io
import shutil
import subprocess
import sys
import zipfile
import kagglehub

PROJECT_DIR = Path("/kaggle/working/Luan-Van-GC-LSTM-GhostNet-CICDDoS2019-v1")
OUTPUT_DIR = PROJECT_DIR / "outputs" / "step2_smoke"
DATA_DIR = Path(kagglehub.dataset_download("dungnguyen28101991/cicddos2019-parquet"))
PROJECT_ARCHIVE_B64 = "..."

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PROJECT_ARCHIVE_B64))) as archive:
    archive.extractall(PROJECT_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
print(f"Project ready at {PROJECT_DIR}")

In [ ]:
command = [
    sys.executable, "-m", "src.step2_smoke",
    "--data-dir", str(DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--config", "configs/base.yaml",
    "--mode-config", "configs/practical_baseline.yaml",
    "--samples-per-file", "2048",
]
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
import json

summary_path = OUTPUT_DIR / "smoke_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
assert summary["status"] == "passed", summary
assert all(run["leakage_status"] == "passed" for run in summary["runs"]), summary
summary